# EBM basado en energía — Denoising Score Matching multiescala en MNIST

Versión modular de `ebm_energia_dsm_mnist`, siguiendo el orden y los módulos de `ebm_denoising.ipynb`. La U-Net aprende una **energía** $E_\theta(x,\sigma)$ y `forward(x, sigma)` devuelve el score $-\nabla_x E_\theta(x,\sigma)$. Se reutilizan los módulos existentes de datos, ruido, entrenamiento y Langevin.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))
print("Project root:", project_root)

import time
import json
import copy
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.utils import save_image
import matplotlib.pyplot as plt

from data import get_mnist_loaders

from ebm.models import EnergyModelDNS
from ebm.sampling import (
    annealed_langevin,
    langevin_at_sigma,
)
from ebm.noise import build_noise_schedule
from ebm.training import train_energy_model_dsm

from utils import (
    plot_image_grid,
    plot_image_evolution,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)

## Configuración

In [ ]:
QUICK_TEST = False

CFG = dict(
    seed            = 42,
    # datos
    batch_size      = 128,
    # niveles de ruido
    sigma_1         = "auto",
    sigma_L         = 0.01,
    overlap_C       = 0.5,
    # muestreo
    T               = 5,
    eps             = "auto",
    # red
    channels        = 64,
    # optimización
    lr              = 1e-4,
    grad_clip       = 1.0,
    ema_decay       = 0.999,
    epochs          = 60,
    max_iters_per_epoch = None,
    # monitorización
    vis_every       = 5,
    ckpt_dir        = str(
        project_root / "checkpoints" / "energy_dsm"
    ),
    pcd_samples_path = str(project_root / "results" / "pcd" / "muestras_pcd_01.pt"),
    score_samples_path = str(project_root / "results" / "dsm" / "muestras_dsm_01.pt"),
    # resultados
    results_dir     = str(
        project_root / "results" / "energy_dsm"
    ),
)

if QUICK_TEST:
    CFG.update(
        epochs=2,
        max_iters_per_epoch=30,
        vis_every=1,
        T=1,
    )

set_seed(CFG["seed"])

IMG_SHAPE = (1, 28, 28)

CFG

## Datos

In [ ]:
train_loader, test_loader = get_mnist_loaders(
    batch_size=CFG["batch_size"],
    normalize=False,
    num_workers=2,
)

x_real, _ = next(iter(train_loader))

print(
    x_real.shape,
    x_real.min().item(),
    x_real.max().item(),
)

In [ ]:
fig, ax = plot_image_grid(
    images=x_real[:32],
    figsize=(8, 4),
    n_cols=8,
)

ax.set_title("MNIST real")
plt.show()

## Niveles de ruido

Se utiliza `build_noise_schedule` igual que en `ebm_denoising`: cálculo automático de $\sigma_1$, niveles, $\gamma$ y paso $\varepsilon$.

In [ ]:
noise_schedule = build_noise_schedule(
    dataset=train_loader.dataset,
    image_shape=IMG_SHAPE,
    device=device,
    sigma_1=CFG["sigma_1"],
    sigma_l=CFG["sigma_L"],
    overlap_c=CFG["overlap_C"],
    n_steps_per_level=CFG["T"],
    eps=CFG["eps"],
)

sigmas = noise_schedule["sigmas"]
sigma_1 = noise_schedule["sigma_1"]
sigma_L = noise_schedule["sigma_l"]
gamma = noise_schedule["gamma"]
L = noise_schedule["n_levels"]
eps = noise_schedule["eps"]
NFE = noise_schedule["nfe"]

print(
    f"distancia máxima ≈ {noise_schedule['dmax']:.2f} | "
    f"mediana ≈ {noise_schedule['dmed']:.2f}"
)

print(
    f"gamma = {gamma:.4f} "
    f"(solapamiento {noise_schedule['overlap']:.2f}) | "
    f"L = {L} niveles"
)

print(
    f"T = {CFG['T']} | "
    f"eps = {eps:.3e} | "
    f"Ec.(4) = {noise_schedule['eq4_ratio']:.3f} "
    "(ideal 1)"
)

print(
    f"evaluaciones de la red por muestra: {NFE}"
)

plt.figure(figsize=(6, 3))
plt.semilogy(
    sigmas.cpu().numpy(),
    ".-",
)
plt.xlabel("nivel i")
plt.ylabel(r"$\sigma_i$")
plt.title("Niveles de ruido")
plt.show()


## Red de energía

`EnergyUNet` está definida en `ebm/models/energy_unet.py`. La arquitectura, el condicionamiento por ruido y la función de energía son los del notebook original. No se define la red de nuevo aquí para mantener el notebook modular.

La energía por imagen es $E_\theta(x,\sigma)$; `model(x, sigma)` obtiene $-\nabla_x E_\theta(x,\sigma)$. Así, el entrenador DSM y los muestreadores reciben la misma interfaz que con `ScoreUNet`. **El cálculo de los gradientes del score respecto a los parámetros implica derivadas de segundo orden y puede aumentar el coste de memoria y tiempo.**

In [ ]:
model = EnergyUNet(
    channels=CFG["channels"],
).to(device)

ema = copy.deepcopy(
    model
).eval()

for parameter in ema.parameters():
    parameter.requires_grad_(False)

print(model)
print(
    "parámetros:",
    sum(
        parameter.numel()
        for parameter
        in model.parameters()
    ),
)


## Muestreador Langevin recocido

Se reutiliza `ebm.sampling.annealed_langevin` porque `EnergyUNet.forward(x, sigma)` devuelve el score.

In [ ]:
sampler_kwargs = dict(
    sigmas=sigmas,
    eps=eps,
    image_shape=IMG_SHAPE,
    n_steps_per_level=CFG["T"],
    device=device,
)

## Entrenamiento

Se reutiliza `train_energy_model_dsm`. El objetivo DSM se aplica al score calculado como el gradiente negativo de la energía.

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CFG["lr"],
)

hist_it = {
    "loss": [],
}

hist_ep = {
    key: []
    for key in [
        "epoch",
        "loss",
        "loss_per_sigma",
        "time_s",
    ]
}

# Mismo ruido inicial para comparar las muestras entre épocas.
generator = torch.Generator(
    device=device
).manual_seed(1234)

vis_init = (
    torch.rand(
        16,
        *IMG_SHAPE,
        generator=generator,
        device=device,
    )
    + sigmas[0]
    * torch.randn(
        16,
        *IMG_SHAPE,
        generator=generator,
        device=device,
    )
)


In [ ]:
hist_it, hist_ep = train_energy_model_dsm(
    model=model,
    ema=ema,
    data_loader=train_loader,
    optimizer=optimizer,
    annealed_sampler=annealed_langevin,
    sigmas=sigmas,
    eps=eps,
    device=device,
    sampler_kwargs=sampler_kwargs,
    config=CFG,
    hist_it=hist_it,
    hist_ep=hist_ep,
    vis_init=vis_init,
)


## Curvas de entrenamiento

In [ ]:
def smooth(v, w=50):
    v = np.asarray(v, dtype=float)
    if len(v) < w: return v
    return np.convolve(v, np.ones(w) / w, mode="valid")


fig, ax = plt.subplots(
    1,
    3,
    figsize=(16, 4),
)

ax[0].plot(
    smooth(
        hist_it["loss"]
    )
)
ax[0].set_yscale("log")
ax[0].set_xlabel("iteración")
ax[0].set_title(
    "Pérdida DSM (media móvil)"
)

ax[1].plot(
    hist_ep["epoch"],
    hist_ep["loss"],
    "o-",
)
ax[1].set_yscale("log")
ax[1].set_xlabel("época")
ax[1].set_title(
    "Pérdida media por época"
)

loss_per_sigma = np.array(
    hist_ep["loss_per_sigma"]
)

sigma_values = (
    sigmas.cpu().numpy()
)

for epoch_index in sorted(
    set([
        0,
        len(loss_per_sigma) // 2,
        len(loss_per_sigma) - 1,
    ])
):
    ax[2].loglog(
        sigma_values,
        loss_per_sigma[epoch_index],
        label=(
            f"época "
            f"{hist_ep['epoch'][epoch_index]}"
        ),
    )

ax[2].invert_xaxis()
ax[2].set_xlabel(r"$\sigma$")
ax[2].set_title(
    "Pérdida por nivel de ruido"
)
ax[2].legend()

plt.tight_layout()
plt.show()


## Generación desde ruido y evolución de la cadena

Cada fila representa una muestra; cada columna muestra el estado tras un nivel de ruido. Se conserva el diagnóstico de ruido efectivo.

In [ ]:
set_seed(123)

levels_to_show = sorted(
    set(
        np.linspace(
            0,
            L - 1,
            8,
        )
        .round()
        .astype(int)
        .tolist()
    )
)

x_final, snapshots, effective_noise = annealed_langevin(
    model=ema,
    n_samples=8,
    seed=123,
    save_levels=levels_to_show,
    **sampler_kwargs,
)

trajectory = torch.stack(
    [
        snapshots[level]
        for level
        in levels_to_show
    ]
    + [x_final],
    dim=-1,
)

_ = plot_image_evolution(
    images=trajectory,
    n_images=8,
    n_intermediate_steps=range(
        len(levels_to_show) + 1
    ),
    figsize=(10, 10),
)

plt.show()

plt.figure(figsize=(7, 3))
plt.semilogx(
    sigmas.cpu().numpy(),
    effective_noise,
    ".-",
)
plt.axhline(
    1,
    color="k",
    ls="--",
    lw=1,
)
plt.gca().invert_xaxis()
plt.xlabel(r"$\sigma_i$")
plt.ylabel("ruido efectivo / σ")
plt.title(
    "Diagnóstico del muestreo "
    "(ideal ≈ 1)"
)
plt.show()


In [ ]:
set_seed(7)

t0 = time.time()

x_many = annealed_langevin(
    model=ema,
    n_samples=64,
    seed=7,
    **sampler_kwargs,
)

print(
    f"64 muestras en "
    f"{time.time() - t0:.1f}s "
    f"({NFE} evaluaciones "
    "de la red por muestra)"
)

fig, ax = plot_image_grid(
    images=x_many,
    figsize=(7, 7),
    n_cols=8,
)

ax.set_title(
    f"64 muestras desde ruido "
    f"(L={L}, T={CFG['T']}, con EMA)"
)

plt.show()


## Experimento: cadena larga

Se continúa la cadena desde muestras generadas durante 2000 pasos en el último nivel de ruido.

In [ ]:
set_seed(11)

long_steps = [
    0,
    100,
    500,
    1000,
    2000,
]

snapshots_long = langevin_at_sigma(
    model=ema,
    x=x_final[:8],
    sigma=sigmas[-1],
    n_steps=max(long_steps),
    step_size=eps,
    save_steps=long_steps,
    device=device,
    seed=11,
)

trajectory_long = torch.stack(
    [
        snapshots_long[step]
        for step
        in long_steps
    ],
    dim=-1,
)

_ = plot_image_evolution(
    images=trajectory_long,
    n_images=8,
    n_intermediate_steps=range(
        len(long_steps)
    ),
    figsize=(10, 10),
)

plt.show()


## Comprobación de ruido

In [ ]:
model_eval = ema
model_eval.eval()

x_real, _ = next(iter(test_loader))
x_real = x_real[:16].to(device)

sigma_value = 0.1

sigma = torch.full(
    (x_real.shape[0],),
    sigma_value,
    device=device,
)

x_noise = torch.rand_like(x_real)

x_corrupted = (
    x_real
    + sigma_value * torch.randn_like(x_real)
)

with torch.no_grad():
    E_real = model_eval.energy(x_real, sigma)
    E_noise = model_eval.energy(x_noise, sigma)
    E_corrupted = model_eval.energy(x_corrupted, sigma)

print("Energía real:", E_real.mean().item())
print("Energía ruido:", E_noise.mean().item())
print("Energía real con ruido:", E_corrupted.mean().item())